In [13]:
import GEOparse
import os 
from pathlib import Path
import re
import pandas as pd
import numpy as np
from skrub import Cleaner

## Target columns of the metadata table

In [14]:
target_columns = ["SRX ID", 
                    "GEO ID", 
                    "Other source ID", 
                    "GSM ID", 
                    "Title", 
                    "Submission date", 
                    "Extract protocol", 
                    "Label protocol", 
                    "Instrument model", 
                    "Organism", 
                    "Strain", 
                    "Genotype", 
                    "Sex", 
                    "Age", 
                    "Tissue", 
                    "Cell type", 
                    "Passages", 
                    "Disease", 
                    "Treatment", 
                    "Contact laboratory", 
                    "Contact department", 
                    "Contact institute", 
                    "Other"]

## Define datasets to download

In [15]:
def read_files_to_dict_basic(directory_path):
    """
    Read all files in directory into dictionary {filename: content}
    """
    file_dict = {}
    
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
    
        if os.path.isfile(file_path):
            try:
                with open(file_path, 'r', encoding='utf-8') as file:
                    content = file.read()
                    file_dict[filename.split('.')[0]] = content.split()
            except Exception as e:
                print(f"Error reading {filename}: {e}")
    
    return file_dict

In [16]:
directory = "PrepareSirt6Metadata/gse_ids/"
files_dict = read_files_to_dict_basic(directory)

## Download the data

In [17]:
species_pheno_data = pd.DataFrame({})

for key in files_dict.keys():
    #print(key)
    for gse_id in files_dict[key]:
        #print(gse_id)
        gse = GEOparse.get_GEO(geo = gse_id, destdir = "./", silent = True)
        pheno_data = gse.phenotype_data
        pheno_data = pheno_data.drop(columns = ['status', 
                           'title', 
                           'last_update_date', 
                           'type', 
                           'channel_count', 
                           'taxid_ch1', 
                           'molecule_ch1', 
                           #'supplementary_file_1', 
                           'data_row_count', 
                           #'characteristics_ch1.0.genotype/variation', 
                           #'characteristics_ch1.1.antibody', 
                           'contact_city', 
                           #'contact_state', 
                           'contact_zip/postal_code', 
                           'contact_country',
                           'contact_name',
                           #'contact_email', 
                           #'contact_phone', 
                           'contact_address',
                           'data_processing',
                           #'library_selection', 
                           #'library_source', 
                           'platform_id'])
        pheno_data = pheno_data.loc[pheno_data.library_strategy == 'RNA-Seq']
        pheno_data['relation'] = pheno_data.relation.str.split('term=', expand=True)[1]
        pheno_data['title'] = gse.metadata['title'][0]
        pheno_data['summary'] = gse.metadata['summary'][0]
        species_pheno_data = pd.concat([species_pheno_data, pheno_data], ignore_index=True)

In [18]:
species_pheno_data.query("organism_ch1 == 'Rattus norvegicus'")

,geo_accession,submission_date,source_name_ch1,organism_ch1,characteristics_ch1.0.tissue,characteristics_ch1.1.development stage,characteristics_ch1.2.group,extract_protocol_ch1,contact_email,contact_institute,...,characteristics_ch1.0.disease state,characteristics_ch1.3.chip antibody,characteristics_ch1.0.growth protocol,characteristics_ch1.1.experiment,characteristics_ch1.2.antibodies,characteristics_ch1.2.transfected with,characteristics_ch1.2.time,characteristics_ch1.4.treatment,characteristics_ch1.1.Sex,characteristics_ch1.0.Stage
289,GSM8498571,Sep 05 2024,primary rat nucleus pulposus cells,Rattus norvegicus,NaN,NaN,NaN,RNA was extracted using RNA easy kit (Qiagen) ...,pranay1130@gmail.com,Thomas Jefferson University,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
290,GSM8498572,Sep 05 2024,primary rat nucleus pulposus cells,Rattus norvegicus,NaN,NaN,NaN,RNA was extracted using RNA easy kit (Qiagen) ...,pranay1130@gmail.com,Thomas Jefferson University,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
291,GSM8498573,Sep 05 2024,primary rat nucleus pulposus cells,Rattus norvegicus,NaN,NaN,NaN,RNA was extracted using RNA easy kit (Qiagen) ...,pranay1130@gmail.com,Thomas Jefferson University,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
292,GSM8498574,Sep 05 2024,primary rat nucleus pulposus cells,Rattus norvegicus,NaN,NaN,NaN,RNA was extracted using RNA easy kit (Qiagen) ...,pranay1130@gmail.com,Thomas Jefferson University,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
293,GSM8498575,Sep 05 2024,primary rat nucleus pulposus cells,Rattus norvegicus,NaN,NaN,NaN,RNA was extracted using RNA easy kit (Qiagen) ...,pranay1130@gmail.com,Thomas Jefferson University,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
294,GSM8498576,Sep 05 2024,primary rat nucleus pulposus cells,Rattus norvegicus,NaN,NaN,NaN,RNA was extracted using RNA easy kit (Qiagen) ...,pranay1130@gmail.com,Thomas Jefferson University,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
295,GSM8498577,Sep 05 2024,primary rat nucleus pulposus cells,Rattus norvegicus,NaN,NaN,NaN,RNA was extracted using RNA easy kit (Qiagen) ...,pranay1130@gmail.com,Thomas Jefferson University,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
296,GSM8498578,Sep 05 2024,primary rat nucleus pulposus cells,Rattus norvegicus,NaN,NaN,NaN,RNA was extracted using RNA easy kit (Qiagen) ...,pranay1130@gmail.com,Thomas Jefferson University,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
len(species_pheno_data.relation.values)

361

In [20]:
species_pheno_data = species_pheno_data.drop(columns=['contact_phone', 
                                                      'library_selection',
                                                      'library_source', 
                                                      'contact_email'])

In [21]:
species_pheno_data = species_pheno_data.loc[:, ~ species_pheno_data.isna().all(axis=0)]

In [22]:
patterns_to_merge = ['Sex', 'Genotype', 'Treatment', 'Cell line', 'Tissue', 'Cell', 'Strain', 'Age']

for pattern in patterns_to_merge:
    target_columns = [item for item in species_pheno_data.columns if pattern.lower() in item.lower()]

    if pattern == "Age":
        patterns_to_remove = ['Passages', 'Agent']

        columns_to_remove = [item for pattern in patterns_to_remove 
                             for item in species_pheno_data.columns 
                             if pattern.lower() in item.lower()]

        target_columns = [item for item in target_columns if item not in columns_to_remove]

    if pattern == "Cell":
        pattern += " type"
    species_pheno_data[pattern] = species_pheno_data[target_columns].apply(lambda x: ' '.join(x.dropna().astype(str)), axis = 1)
    species_pheno_data = species_pheno_data.drop(columns = target_columns)

#species_pheno_data = species_pheno_data.replace(r'^\s*$', np.nan, regex = True)

In [23]:
species_pheno_data.columns

Index(['geo_accession', 'submission_date', 'source_name_ch1', 'organism_ch1',
       'characteristics_ch1.2.group', 'extract_protocol_ch1',
       'contact_institute', 'instrument_model', 'library_strategy', 'relation',
       'supplementary_file_1', 'series_id', 'title', 'summary',
       'growth_protocol_ch1', 'contact_state',
       'characteristics_ch1.2.passages', 'contact_department', 'description',
       'contact_laboratory', 'characteristics_ch1.4.batch',
       'characteristics_ch1.3.agent', 'characteristics_ch1.3.a6 status',
       'characteristics_ch1.4.cd34+ status',
       'characteristics_ch1.0.disease state',
       'characteristics_ch1.2.transfected with', 'characteristics_ch1.2.time',
       'Sex', 'Genotype', 'Treatment', 'Tissue', 'Cell type', 'Strain', 'Age'],
      dtype='str')

In [24]:
genotype_patterns = ['characteristics_ch1.2.transfected with', 'Genotype', 'characteristics_ch1.2.group']
treatment_patterns = ['Treatment', 'characteristics_ch1.3.agent']

In [25]:
columns_to_merge = [item for pattern in genotype_patterns  
                     for item in species_pheno_data.columns 
                     if pattern.lower() in item.lower()]

species_pheno_data['Genotype'] = species_pheno_data[columns_to_merge].apply(lambda x: ' '.join(x.dropna().astype(str)), axis = 1)
columns_to_drop = [item for item in genotype_patterns if item != 'Genotype']
species_pheno_data = species_pheno_data.drop(columns = columns_to_drop)

In [26]:
columns_to_merge = [item for pattern in treatment_patterns  
                     for item in species_pheno_data.columns 
                     if pattern.lower() in item.lower()]

species_pheno_data['Treatment'] = species_pheno_data[columns_to_merge].apply(lambda x: ' '.join(x.dropna().astype(str)), axis = 1)
columns_to_drop = [item for item in treatment_patterns if item != 'Treatment']
species_pheno_data = species_pheno_data.drop(columns = columns_to_drop)

In [27]:
columns_to_drop = ['supplementary_file_1', 
                   'source_name_ch1', 
                   'contact_state', 
                   'contact_laboratory', 
                   'contact_department', 
                   #'description'
                  ]
species_pheno_data = species_pheno_data.drop(columns = columns_to_drop)

In [28]:
pattern = '_ch1'
cleaned_colnames = species_pheno_data.columns.str.split('.').str[-1]
species_pheno_data.columns = [re.sub(rf'{pattern}', '', item) for item in cleaned_colnames]

In [29]:
species_pheno_data.columns

Index(['geo_accession', 'submission_date', 'organism', 'extract_protocol',
       'contact_institute', 'instrument_model', 'library_strategy', 'relation',
       'series_id', 'title', 'summary', 'growth_protocol', 'passages',
       'description', 'batch', 'a6 status', 'cd34+ status', 'disease state',
       'time', 'Sex', 'Genotype', 'Treatment', 'Tissue', 'Cell type', 'Strain',
       'Age'],
      dtype='str')

In [30]:
condition_patterns = ['a6 status', 'cd34+ status', 'disease state']

columns_to_merge = [item for pattern in condition_patterns  
                     for item in species_pheno_data.columns 
                     if pattern.lower() in item.lower()]

species_pheno_data[columns_to_merge].apply(lambda x: ' '.join(x.dropna().astype(str)), axis = 1).unique()

species_pheno_data['Condition'] = species_pheno_data[columns_to_merge].apply(lambda x: ' '.join(x.dropna().astype(str)), axis = 1)
species_pheno_data = species_pheno_data.drop(columns = columns_to_merge)

In [31]:
#species_pheno_data = species_pheno_data.replace(r'^\s*$', np.nan, regex=True)

In [32]:
species_pheno_data.columns

Index(['geo_accession', 'submission_date', 'organism', 'extract_protocol',
       'contact_institute', 'instrument_model', 'library_strategy', 'relation',
       'series_id', 'title', 'summary', 'growth_protocol', 'passages',
       'description', 'batch', 'time', 'Sex', 'Genotype', 'Treatment',
       'Tissue', 'Cell type', 'Strain', 'Age', 'Condition'],
      dtype='str')

In [33]:
species_pheno_data.columns = [name[0].upper() + name[1:] for name in species_pheno_data.columns.str.lower()]

### Remove whitespaces

In [34]:
string_columns = species_pheno_data.select_dtypes(include=['object']).columns
species_pheno_data[string_columns] = species_pheno_data[string_columns].apply(lambda x: x.str.strip())

/var/folders/kb/2390td2n4f3bqhz9pvyr26w00000gn/T/ipykernel_29931/1331201016.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = species_pheno_data.select_dtypes(include=['object']).columns


## Cleaning the metadata

In [35]:
species_pheno_data = (
    species_pheno_data
    .assign(Condition=lambda df: np.where(
        df.Genotype == 'Myeloid Sirt6 KO',
        df.Description.str.split(',').str[-1],
        df.Condition 
    ))
)

Drosophila dataset

In [36]:
species_pheno_data = species_pheno_data.query("Description not in ['IECs from WT mice', 'IECs from TgSTAT6vt mice']")

genotypes_to_drop = ['tubulinGeneSwitch>w1118; w1118', 'tubulinGeneSwitch>w1118; UAS-dMyc']
species_pheno_data = species_pheno_data.query(rf"Genotype not in {genotypes_to_drop}")

GSE212057

In [37]:
species_pheno_data = (
    species_pheno_data
    .assign(Genotype = lambda df: np.where(
        df.Series_id == 'GSE212057',
        df.Genotype.replace({'WT': 'SIRT6-OE', 'Ctr': 'WT', 'K3Q': 'SIRT6-OE/K3Q', 'K3R': 'SIRT6-OE/K3R'}),
        df.Genotype
    ))
)

In [38]:
species_pheno_data = (
    species_pheno_data
    .assign(Batch = lambda df: np.where(
        df.Series_id == 'GSE212057',
        df.Description.str.split(',').str[1],
        df.Batch
    ))
)

Mdx mouse datasets (GSE168984, GSE168983)

In [39]:
species_pheno_data = (
    species_pheno_data
    .assign(
        Condition=lambda df: np.where(
            df.Series_id.isin(['GSE168984', 'GSE168983']),
            np.where(
                df.Genotype.isin(["mdx", "mdx, Sirt6-mKO"]),
                "mdx",
                "Control"
            ),
            df.Condition
        ),
        Tissue=lambda df: np.select(
            [
                (df.Series_id == 'GSE168984') & (df.Tissue == ''),
                (df.Series_id == 'GSE168983') & (df.Tissue == ''),
            ],
            ['Muscle', 'MuSCs'],
            df.Tissue
        ),
        Genotype=lambda df: np.where(
            df.Series_id.isin(['GSE168984', 'GSE168983']),
            df.Genotype.replace({'mdx': 'WT', 'mdx, Sirt6-mKO': 'SIRT6-KO'}),
            df.Genotype
        )
    )
)

Remove samples with Atm-/- genotype (GSE109280)

In [40]:
species_pheno_data = species_pheno_data.query("Genotype != 'Atm-/-' & Treatment != 'Wild-type Glucose deprivation'")

Handling whitespaces in Genotype column

In [41]:
(species_pheno_data
    .query("Genotype == ''")
    .Series_id.value_counts()
)

Series_id
GSE102813              18
GSE235082              12
GSE246209               6
GSE130690,GSE130692     6
Name: count, dtype: int64

In [42]:
species_pheno_data = (
    species_pheno_data
    .assign(Genotype = lambda df: np.select(
            [
                df['Cell type'] == 'L-C-B',
                df['Cell type'] == 'SIRT6.2-7',
                df['Cell type'] == 'SIRT6.1-1'
            ],
            ['WT', 'SIRT6-Het', 'SIRT6-KO'],
            default = df.Genotype
            ),
    )
    .assign(**{'Cell type': lambda df: np.where(
        df.Series_id == 'GSE102813',
        'SK-MEL-239',
        df['Cell type']
    )})
)

In [43]:
species_pheno_data = (
    species_pheno_data
    .assign(Genotype = lambda df: np.where(
        df.Series_id == 'GSE235082',
        df.Treatment.str.split(' Chondrocytes').str[0].replace({'siCtrl': 'WT', 'siSirt6': 'SIRT6-KO'}),
        df.Genotype
    ))
)

In [44]:
species_pheno_data = (
    species_pheno_data
    .assign(Genotype = lambda df: np.where(
        df.Series_id == 'GSE246209',
        df.Treatment.replace({'Control': 'WT', 'siSirt6': 'SIRT6-KO'}),
        df.Genotype
    ))
)

In [45]:
species_pheno_data = (
    species_pheno_data
    .assign(Genotype = lambda df: np.where(
        df.Series_id == 'GSE130690,GSE130692',
        df.Treatment.replace({'Wild-type': 'WT', 'SIRT6-Knock Out': 'SIRT6-KO'}),
        df.Genotype
    ))
)

In [46]:
new_ages = 4*['10 months'] + 6*['16 months']
mask = species_pheno_data['Series_id'] == 'GSE287696'
species_pheno_data.loc[mask, 'Age'] = new_ages

In [47]:
species_pheno_data = (
    species_pheno_data
    .assign(Organism=lambda df: df.Organism.replace({'Mus': 'Mus musculus'}))
)

In [48]:
species_pheno_data.loc[species_pheno_data["Organism"] == "Sus scrofa"]

,Geo_accession,Submission_date,Organism,Extract_protocol,Contact_institute,Instrument_model,Library_strategy,Relation,Series_id,Title,...,Batch,Time,Sex,Genotype,Treatment,Tissue,Cell type,Strain,Age,Condition
0,GSM4889180,Nov 09 2020,Sus scrofa,The cell was lysised and the RNA was amplifica...,"College of Animal Science and Technology, Nanj...",HiSeq X Ten,RNA-Seq,SRX9461172,GSE161068,SIRT6 maintains redox homeostasis to promote p...,...,NaN,NaN,,control,,oocyte,,,Metaphase II,
1,GSM4889181,Nov 09 2020,Sus scrofa,The cell was lysised and the RNA was amplifica...,"College of Animal Science and Technology, Nanj...",HiSeq X Ten,RNA-Seq,SRX9461173,GSE161068,SIRT6 maintains redox homeostasis to promote p...,...,NaN,NaN,,control,,oocyte,,,Metaphase II,
2,GSM4889182,Nov 09 2020,Sus scrofa,The cell was lysised and the RNA was amplifica...,"College of Animal Science and Technology, Nanj...",HiSeq X Ten,RNA-Seq,SRX9461174,GSE161068,SIRT6 maintains redox homeostasis to promote p...,...,NaN,NaN,,sirt6 inhibited,,oocyte,,,Metaphase II,
3,GSM4889183,Nov 09 2020,Sus scrofa,The cell was lysised and the RNA was amplifica...,"College of Animal Science and Technology, Nanj...",HiSeq X Ten,RNA-Seq,SRX9461171,GSE161068,SIRT6 maintains redox homeostasis to promote p...,...,NaN,NaN,,sirt6 inhibited,,oocyte,,,Metaphase II,


In [49]:
species_pheno_data = (
    species_pheno_data
    .assign(Condition=lambda df: np.where(
        df.Geo_accession.isin(['GSM6845112', 'GSM6845118', 'GSM6845124']),
        0,
        df.Condition 
    ))
    .assign(Condition=lambda df: np.where(
        df.Geo_accession.isin(['GSM6845113', 'GSM6845119', 'GSM6845125']),
        20,
        df.Condition 
    ))
    .assign(Condition=lambda df: np.where(
        df.Geo_accession.isin(['GSM6845114', 'GSM6845120', 'GSM6845126']),
        60,
        df.Condition 
    ))
    .assign(Treatment=lambda df: np.where(
        df.Geo_accession.isin(['GSM6845112', 'GSM6845118', 'GSM6845124', 'GSM6845115', 'GSM6845118', 'GSM6845121', 'GSM6845124', 'GSM6845127']),
        'LPS',
        df.Treatment 
    ))
)

In [52]:
species_pheno_data = (
    species_pheno_data
    .assign(Genotype = lambda df: np.where(
        df.Series_id == 'GSE213425',
        df.Genotype.replace({'Control': 'WT', 'SIRT6': 'SIRT6-OE'}),
        df.Genotype
    ))
)

In [ ]:
species_pheno_data = (
    species_pheno_data
    .assign(Condition=lambda df: np.where(
        df.Geo_accession.isin(['GSM1576142', 'GSM1576143', 'GSM1576144', 'GSM1576145']),
        "Early",
        df.Condition 
    ))
    .assign(Condition=lambda df: np.where(
        df.Geo_accession.isin(['GSM1576146', 'GSM1576147', 'GSM1576148', 'GSM1576149']),
        "Late",
        df.Condition 
    ))
)

In [63]:
species_pheno_data.query("Series_id == 'GSE64642'")

,Geo_accession,Submission_date,Organism,Extract_protocol,Contact_institute,Instrument_model,Library_strategy,Relation,Series_id,Title,...,Batch,Time,Sex,Genotype,Treatment,Tissue,Cell type,Strain,Age,Condition
191,GSM1576142,Jan 04 2015,Homo sapiens,Total RNA was extracted from the cells using t...,"Biodynamics Optical Imaging Center (BIOPIC), P...",Illumina HiSeq 2000,RNA-Seq,SRX827480,GSE64642,SIRT6 regulates redox homeostasis in human mes...,...,NaN,NaN,,WT; SIRT6+/+,We firstly generated isogenic human embryonic ...,,human mesenchymal stem cell (hMSCs) H9,,,Early
192,GSM1576143,Jan 04 2015,Homo sapiens,Total RNA was extracted from the cells using t...,"Biodynamics Optical Imaging Center (BIOPIC), P...",Illumina HiSeq 2000,RNA-Seq,SRX827481,GSE64642,SIRT6 regulates redox homeostasis in human mes...,...,NaN,NaN,,WT; SIRT6+/+,We firstly generated isogenic human embryonic ...,,human mesenchymal stem cell (hMSCs) H9,,,Early
193,GSM1576144,Jan 04 2015,Homo sapiens,Total RNA was extracted from the cells using t...,"Biodynamics Optical Imaging Center (BIOPIC), P...",Illumina HiSeq 2000,RNA-Seq,SRX827482,GSE64642,SIRT6 regulates redox homeostasis in human mes...,...,NaN,NaN,,KO; SIRT6-/-,We firstly generated isogenic human embryonic ...,,human mesenchymal stem cell (hMSCs) H9,,,Early
194,GSM1576145,Jan 04 2015,Homo sapiens,Total RNA was extracted from the cells using t...,"Biodynamics Optical Imaging Center (BIOPIC), P...",Illumina HiSeq 2000,RNA-Seq,SRX827483,GSE64642,SIRT6 regulates redox homeostasis in human mes...,...,NaN,NaN,,KO; SIRT6-/-,We firstly generated isogenic human embryonic ...,,human mesenchymal stem cell (hMSCs) H9,,,Early
195,GSM1576146,Jan 04 2015,Homo sapiens,Total RNA was extracted from the cells using t...,"Biodynamics Optical Imaging Center (BIOPIC), P...",Illumina HiSeq 2000,RNA-Seq,SRX827484,GSE64642,SIRT6 regulates redox homeostasis in human mes...,...,NaN,NaN,,WT; SIRT6+/+,We firstly generated isogenic human embryonic ...,,human mesenchymal stem cell (hMSCs) H9,,,Late
196,GSM1576147,Jan 04 2015,Homo sapiens,Total RNA was extracted from the cells using t...,"Biodynamics Optical Imaging Center (BIOPIC), P...",Illumina HiSeq 2000,RNA-Seq,SRX827485,GSE64642,SIRT6 regulates redox homeostasis in human mes...,...,NaN,NaN,,WT; SIRT6+/+,We firstly generated isogenic human embryonic ...,,human mesenchymal stem cell (hMSCs) H9,,,Late
197,GSM1576148,Jan 04 2015,Homo sapiens,Total RNA was extracted from the cells using t...,"Biodynamics Optical Imaging Center (BIOPIC), P...",Illumina HiSeq 2000,RNA-Seq,SRX827486,GSE64642,SIRT6 regulates redox homeostasis in human mes...,...,NaN,NaN,,KO; SIRT6-/-,We firstly generated isogenic human embryonic ...,,human mesenchymal stem cell (hMSCs) H9,,,Late
198,GSM1576149,Jan 04 2015,Homo sapiens,Total RNA was extracted from the cells using t...,"Biodynamics Optical Imaging Center (BIOPIC), P...",Illumina HiSeq 2000,RNA-Seq,SRX827487,GSE64642,SIRT6 regulates redox homeostasis in human mes...,...,NaN,NaN,,KO; SIRT6-/-,We firstly generated isogenic human embryonic ...,,human mesenchymal stem cell (hMSCs) H9,,,Late


### Standartize genotype names

In [64]:
replacement_wt = ['wild-type', 
                  'wild type', 
                  'wildtype', 
                  'SIRT6Ctrl/ShCtrl', 
                  'Ctr', 
                  'control',
                  'LoxP',
                  'Wild type',
                  'WT; SIRT6+/+', 
                  'Delta16HER2', 
                  'w1118',
                  'tub-gal4>w1118 (background control for dSirt6 overexpression)']

In [65]:
replacement_ko = ['SIRT6 mutant', 
                  'Sirt6 KO', 
                  'SIRT6KD/ShSirt6', 
                  'Sirt6-/-', 
                  'Sirt6 liver-specific knockout', 
                  'SIRT6 KO', 'SIRT6 KO (K14-cre; SIRT6 flox/flox)', 
                  'KO; SIRT6-/-', 
                  'brain-specific SIRT6 knockout', 
                  'Sirt6 IEC specific knockout', 
                  'Sirt6 cKO', 
                  'Sirt6 knockout', 
                  'Sirt6mKO', 
                  'sirt6 inhibited',
                  'SIRT6', 
                  'Myeloid Sirt6 KO']

In [66]:
replacement_oe = ['Delta16HER2/SIRT6-OE', 'tub-gal4>UAS-dSirt6', 'SIRT6-transgenic', 'SIRT6 transgenic']
replacement_double_oe =['tubulinGeneSwitch> UAS-dSirt6; UAS-dMyc']

In [67]:
species_pheno_data['Genotype'] = species_pheno_data['Genotype'].replace(replacement_wt, 'WT')
species_pheno_data['Genotype'] = species_pheno_data['Genotype'].replace(replacement_ko, 'SIRT6-KO')
species_pheno_data['Genotype'] = species_pheno_data['Genotype'].replace(replacement_oe, 'SIRT6-OE')
species_pheno_data['Genotype'] = species_pheno_data['Genotype'].replace(replacement_double_oe, 'SIRT6-OE/MYC-OE')

In [68]:
species_pheno_data['Genotype'] = species_pheno_data['Genotype'].replace({'Chondrocytes were nucleofected with siRNA to Sirt6 or an empty vector control (AMAXA nucleofection method) for 72 hours siCtrl': 'WT', 
                                                                         'Chondrocytes were nucleofected with siRNA to Sirt6 or an empty vector control (AMAXA nucleofection method) for 72 hours siSirt6': 'SIRT6-KO'})

In [69]:
species_pheno_data['Genotype'].value_counts()

Genotype
WT                 170
SIRT6-KO           119
SIRT6-OE            36
SIRT6-OE/K3Q         6
SIRT6-OE/K3R         6
SIRT6-Het            6
SIRT6-OE/MYC-OE      2
Name: count, dtype: int64

In [70]:
species_pheno_data.Treatment = species_pheno_data.Treatment.replace(r'^\s*$', "Control", regex = True)

In [71]:
species_pheno_data.Condition = species_pheno_data.Condition.replace(r'^\s*$', "Control", regex = True)

In [72]:
species_pheno_data = species_pheno_data.drop(columns=['Description'])

In [73]:
species_pheno_data.Condition.value_counts()

Condition
Control                     276
BRAF metastatic melanoma     18
mdx                          10
a6High CD34p                  5
a6High CD34m                  5
a6Low CD34m                   5
Early                         4
Late                          4
0                             3
20                            3
60                            3
 0                            3
 20                           3
 60                           3
Name: count, dtype: int64

In [74]:
species_pheno_data.Tissue.value_counts()

Tissue
                                    121
liver                                48
aorta                                25
brain                                20
Fat body                             16
Head                                 12
muscle                               12
pancreatic islets                    10
Muscle                                9
thymus                                8
heart                                 8
lung                                  8
kidney                                8
murine embryonic fibroblast cell      6
knee cartilage                        6
Jejunum                               6
MuSCs                                 6
oocyte                                4
breast cancer tumor                   4
Thymus                                4
Umbilical Vein                        4
Name: count, dtype: int64

In [75]:
species_pheno_data['Cell type'].value_counts()

Cell type
                                                             192
BMMC                                                          18
SK-MEL-239                                                    18
Skin tumor cells                                              15
non-small cell lung carcinoma cell line H1299                 12
non-small cell lung carcinoma cell line A549                  12
Primary isolated human chondrocytes                           12
alpha, beta, delta, and other islet cells                     10
human mesenchymal stem cell (hMSCs) H9                         8
nuclues pulposus cells primary rat nucleus pulposus cells      8
Vascular smooth muscle cells VSMCs                             6
neural stem cells                                              6
Embryonic stem cells                                           6
chondrocytes                                                   6
primarily isolated Intestinal epithelial cells                 6
Muscle stem cel

In [76]:
species_pheno_data['Organism'].value_counts()

Organism
Mus musculus               175
Homo sapiens                66
Macaca fascicularis         64
Drosophila melanogaster     28
Rattus norvegicus            8
Sus scrofa                   4
Name: count, dtype: int64

In [77]:
species_pheno_data.columns

Index(['Geo_accession', 'Submission_date', 'Organism', 'Extract_protocol',
       'Contact_institute', 'Instrument_model', 'Library_strategy', 'Relation',
       'Series_id', 'Title', 'Summary', 'Growth_protocol', 'Passages', 'Batch',
       'Time', 'Sex', 'Genotype', 'Treatment', 'Tissue', 'Cell type', 'Strain',
       'Age', 'Condition'],
      dtype='str')

In [78]:
species_pheno_data.groupby(['Organism', 'Genotype']).size()

Organism                 Genotype       
Drosophila melanogaster  SIRT6-KO            6
                         SIRT6-OE            7
                         SIRT6-OE/MYC-OE     2
                         WT                 13
Homo sapiens             SIRT6-Het           6
                         SIRT6-KO           16
                         SIRT6-OE            8
                         SIRT6-OE/K3Q        6
                         SIRT6-OE/K3R        6
                         WT                 24
Macaca fascicularis      SIRT6-KO           28
                         WT                 36
Mus musculus             SIRT6-KO           63
                         SIRT6-OE           21
                         WT                 91
Rattus norvegicus        SIRT6-KO            4
                         WT                  4
Sus scrofa               SIRT6-KO            2
                         WT                  2
dtype: int64

## Add chinese data

In [79]:
default_value = [""] * 6

In [80]:
dev_cell_data = dict.fromkeys(species_pheno_data.columns, default_value)

In [81]:
dev_cell_data

{'Geo_accession': ['', '', '', '', '', ''],
 'Submission_date': ['', '', '', '', '', ''],
 'Organism': ['', '', '', '', '', ''],
 'Extract_protocol': ['', '', '', '', '', ''],
 'Contact_institute': ['', '', '', '', '', ''],
 'Instrument_model': ['', '', '', '', '', ''],
 'Library_strategy': ['', '', '', '', '', ''],
 'Relation': ['', '', '', '', '', ''],
 'Series_id': ['', '', '', '', '', ''],
 'Title': ['', '', '', '', '', ''],
 'Summary': ['', '', '', '', '', ''],
 'Growth_protocol': ['', '', '', '', '', ''],
 'Passages': ['', '', '', '', '', ''],
 'Batch': ['', '', '', '', '', ''],
 'Time': ['', '', '', '', '', ''],
 'Sex': ['', '', '', '', '', ''],
 'Genotype': ['', '', '', '', '', ''],
 'Treatment': ['', '', '', '', '', ''],
 'Tissue': ['', '', '', '', '', ''],
 'Cell type': ['', '', '', '', '', ''],
 'Strain': ['', '', '', '', '', ''],
 'Age': ['', '', '', '', '', ''],
 'Condition': ['', '', '', '', '', '']}

In [82]:
Extract_protocol = ['Total RNA was extracted using TRIzol (Thermo Fisher Scientific), reverse transcribed into cDNA using GoScript reverse transcription system (Promega), and genomic DNA was removed using DNA free kit (Thermo Fisher Scientific). The genomic DNA itself was separated from the DNA extraction kit (TIANGEN). PCR was performed with PrimeSTAR polymerase. RT–qPCR was performed in CFX384 real-time system (Bio-Rad) using THUNDERBIRD SYBR qPCR Mix (TOYOBO).']

In [83]:
Summary = ['Here, we integrated epigenomic, three-dimensional genome, and transcriptomic analyses of Sirtuins-deficient hMPCs to identify molecular pathways involved in human stem cell aging. We found that the chromatin states of hMPCs after Sirtuins deficiency tend to shift to an activated state with a broad spectrum of increased acetylation levels, and these increases are mostly distributed in the distal end. From the chromatin high-level structure, SIRT1-7 deficiency triggers common epigenetic alterations, as evidenced by increased TAD internal interactions, enhanced loop interactions, and enrichment of activating chromatin signals. Our research uses 3D chromatin organization landscape of human stem cells to establish the connections between Sirtuins and their putative target effector genes, founding the placental specific gene PAPPA was a potential driving force for regulating the aging process.']

In [84]:
Growth_protocol = ['CRISPR/Cas9-mediated gene editing was conducted following established protocols.90 Prior to the electroporation, WT hESCs were cultured on Matrigel-coated plates and pretreated with the ROCK inhibitor Y-27632 (Selleck). Electroporation was carried out using 4D-Nucleofector (Lonza), and vectors (Addgene #87110) containing sgRNAs targeting SIRT1, SIRT4, or SIRT5 along with Cas9 plasmid (Addgene #87109) were introduced into the cells simultaneously. Subsequently, the cells were cultured on Matrigel-coated plates with mTESR supplemented with Y-27632. GFP/mCherry double-positive cells were purified by fluorescence-activated cell sorting (FACS). The generation of SIRT2-/-, SIRT3-/-, SIRT6-/- and SIRT7-/- hESCs was as described.']

In [85]:
Title = ['Sirtuins modulate sub-TAD organization and repress placenta-specific genes that drive cellular aging']

In [86]:
dev_cell_data['Geo_accession'] = ['HRA003336'] * 6
dev_cell_data['Series_id'] = ['HRA003336'] * 6 
dev_cell_data['Submission_date'] = ['Aug 31 2023'] * 6
dev_cell_data['Organism'] = ['Homo sapiens'] * 6
dev_cell_data['Extract_protocol'] = Extract_protocol * 6
dev_cell_data['Contact_institute'] = ['Institute of Zoology, Chinese Academy of Sciences'] * 6
dev_cell_data['Instrument_model'] = ['HiSeq X Ten'] * 6
dev_cell_data['Library_strategy'] = ['RNA-Seq'] * 6
dev_cell_data['Relation'] = ['HRR1202737', 'HRR1202738', 'HRR1202739', 'HRR1202719', 'HRR1202720', 'HRR1202721']
dev_cell_data['Title'] = Title * 6
dev_cell_data['Summary'] = Summary * 6
dev_cell_data['Growth_protocol'] = Growth_protocol * 6
dev_cell_data['Genotype'] = ['WT', 'WT', 'WT', "SIRT6-KO", "SIRT6-KO", "SIRT6-KO"]
dev_cell_data['Cell type'] = ['hMSCs'] * 6
dev_cell_data['Condition'] = ['Control'] * 6
dev_cell_data['Treatment'] = ['Control'] * 6

In [87]:
dev_cell_data_df = pd.DataFrame.from_dict(dev_cell_data)

In [88]:
species_pheno_data = pd.concat([species_pheno_data, dev_cell_data_df]).reset_index(drop = True)

In [89]:
species_pheno_data['Cell type'].unique()

<ArrowStringArray>
[                                                         '',
                        'Vascular smooth muscle cells VSMCs',
                                         'neural stem cells',
                                      'Embryonic stem cells',
                                                      'BMMC',
                                              'chondrocytes',
                 'alpha, beta, delta, and other islet cells',
            'primarily isolated Intestinal epithelial cells',
                                         'Muscle stem cells',
                                          'Skin tumor cells',
                    'human mesenchymal stem cell (hMSCs) H9',
             'non-small cell lung carcinoma cell line H1299',
              'non-small cell lung carcinoma cell line A549',
                                                'SK-MEL-239',
                                         'endothelial cells',
                       'Primary isolated human chon

In [90]:
species_pheno_data.columns

Index(['Geo_accession', 'Submission_date', 'Organism', 'Extract_protocol',
       'Contact_institute', 'Instrument_model', 'Library_strategy', 'Relation',
       'Series_id', 'Title', 'Summary', 'Growth_protocol', 'Passages', 'Batch',
       'Time', 'Sex', 'Genotype', 'Treatment', 'Tissue', 'Cell type', 'Strain',
       'Age', 'Condition'],
      dtype='str')

In [91]:
new_columns = [col.replace(' ', '_').lower() for col in species_pheno_data.columns]

In [92]:
new_columns[0] = 'accession'

In [93]:
species_pheno_data.columns = new_columns

In [94]:
species_pheno_data.columns

Index(['accession', 'submission_date', 'organism', 'extract_protocol',
       'contact_institute', 'instrument_model', 'library_strategy', 'relation',
       'series_id', 'title', 'summary', 'growth_protocol', 'passages', 'batch',
       'time', 'sex', 'genotype', 'treatment', 'tissue', 'cell_type', 'strain',
       'age', 'condition'],
      dtype='str')

In [95]:
species_pheno_data['strain'].unique()

<ArrowStringArray>
['', '129 SvJae F1', 'C57BL/6', 'C57BL/6J', 'C57BL/6JOlaHsd', 'w1118']
Length: 6, dtype: str

In [96]:
species_pheno_data = (
    species_pheno_data
    .assign(treatment=lambda df: np.where(
        df.accession.isin([f'GSM{i}' for i in range(5743582, 5743590)]),
        'Adult (10 days)',
        df.treatment 
    ))
    .assign(treatment=lambda df: np.where(
        df.accession.isin([f'GSM{i}' for i in range(5743590, 5743596)]),
        'Aged (40 days)',
        df.treatment 
    ))
    .assign(age=lambda df: np.where(
        df.series_id.isin(['GSE191320']),
        np.nan,
        df.age 
    ))   
)

## Date conversion

In [97]:
species_pheno_data['submission_date'] = pd.to_datetime(species_pheno_data['submission_date'], 
                                                       format='%b %d %Y', 
                                                       errors='coerce')

## Clean dataset

In [98]:
pheno_data_clean = Cleaner(drop_null_fraction = 1,
                       drop_if_constant = False, 
                       drop_if_unique = False, 
                       datetime_format = None, 
                       numeric_dtype = None, 
                       n_jobs=1).fit_transform(species_pheno_data)

In [99]:
pheno_data_clean['cell_type'].unique()

<ArrowStringArray>
[                                                        nan,
                        'Vascular smooth muscle cells VSMCs',
                                         'neural stem cells',
                                      'Embryonic stem cells',
                                                      'BMMC',
                                              'chondrocytes',
                 'alpha, beta, delta, and other islet cells',
            'primarily isolated Intestinal epithelial cells',
                                         'Muscle stem cells',
                                          'Skin tumor cells',
                    'human mesenchymal stem cell (hMSCs) H9',
             'non-small cell lung carcinoma cell line H1299',
              'non-small cell lung carcinoma cell line A549',
                                                'SK-MEL-239',
                                         'endothelial cells',
                       'Primary isolated human chon

In [100]:
pheno_data_clean['cell_type'] = pheno_data_clean.cell_type.replace({'human mesenchymal stem cell (hMSCs) H9': 'hMSCs', 
                                    'non-small cell lung carcinoma cell line H1299': 'H1299', 
                                    'Primary isolated human chondrocytes': 'chondrocytes', 
                                    'non-small cell lung carcinoma cell line A549': 'A549', 
                                    'Vascular smooth muscle cells VSMCs': 'VSMCs', 
                                    'Muscle stem cells': 'MuSCs', 
                                    'Embryonic stem cells': 'hESCs', 
                                    'primarily isolated Intestinal epithelial cells': 'IEC', 
                                    'nuclues pulposus cells primary rat nucleus pulposus cells': 'nucleus pulposus cells'
                                   })

In [101]:
pheno_data_clean = (
    pheno_data_clean
    .assign(cell_type = lambda df: np.where(
        df.series_id == 'GSE168983',
        df.tissue,
        df.cell_type
    ))
    .assign(tissue = lambda df: np.where(
        df.series_id == 'GSE168983',
        df.tissue.replace({'MuSCs': np.nan}),
        df.tissue
    ))
)#.query("series_id == 'GSE168983'")

In [102]:
pheno_data_clean = (
    pheno_data_clean
    .assign(cell_type = lambda df: np.where(
        df.series_id == 'GSE161068',
        df.tissue,
        df.cell_type
    ))
    .assign(tissue = lambda df: np.where(
        df.series_id == 'GSE161068',
        df.tissue.replace({'oocyte': np.nan}),
        df.tissue
    ))
)

In [103]:
pheno_data_clean = (
    pheno_data_clean
    .assign(cell_type = lambda df: np.where(
        df.series_id == 'GSE109280',
        df.tissue.replace({'murine embryonic fibroblast cell': 'MEFs'}),
        df.cell_type
    ))
    .assign(tissue = lambda df: np.where(
        df.series_id == 'GSE109280',
        df.tissue.replace({'murine embryonic fibroblast cell': np.nan}),
        df.tissue
    ))
)

In [104]:
pheno_data_clean = (
    pheno_data_clean
    .assign(condition = lambda df: np.where(
        df.series_id == 'GSE216185,GSE216186',
        df.tissue.replace({'breast cancer tumor': 'Breast cancer tumor'}),
        df.condition
    ))
    .assign(tissue = lambda df: np.where(
        df.series_id == 'GSE216185,GSE216186',
        df.tissue.replace({'breast cancer tumor': np.nan}),
        df.tissue
    ))
)

In [105]:
pheno_data_clean = (
    pheno_data_clean
    .assign(condition = lambda df: np.where(
        df.series_id == 'GSE235082',
        df.time,
        df.condition
    ))
    .assign(time = lambda df: np.where(
        df.series_id == 'GSE235082',
        df.time.replace({'72 hours': np.nan}),
        df.time
    ))
)

In [106]:
pheno_data_clean = pheno_data_clean.drop(columns = ['time'])

In [107]:
pheno_data_clean['batch'] = pheno_data_clean.batch.replace({'batch 2': '2',
                                                            'batch 1': '1',
                                                            '2nd': '2',
                                                            '1st': '1' })

In [108]:
pheno_data_clean['treatment'].unique() #str.capitalize()

<ArrowStringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                          'Control',
                                                                                                                                                                                                                                                                                                                                                                                                                                                                           'siSirt6',
                                         

In [109]:
pheno_data_clean['treatment'] = pheno_data_clean.treatment.replace({'Monkey tissues were homogenized in liquid nitrogen, then RNA were extracted using TRIzol reagent protocol, and purified using RNeasy Mini Kit (Qiagen) according to the manuals. After quantification of RNA by Fragment Analyzer (Advanced Analytical), 1.5 μg of total RNA was used to construct sequencing libraries by TruSeq RNA Sample Preparation Kit (Illumina) following the manufacturer’s standard protocol. The libraries were sequenced on the HiSeq X Ten platform.': np.nan, 
                                    'DMSO TDO2 inhibitor 680C91 (Tocris Bioscience, Bristol, UK) was dissolved in dimethyl sulfoxide (DMSO). TDO2 inhibitor / DMSO were added to the yeast extract in the diet to a final concentration of 100 µM (680C91) . Flies were introduced to the inhibitor-supplemented media upon eclosion; supplemented media were changed every other day. Files were snaped freeze in liquid nitrogen at the age of 21 days for further process.': 'DMSO TDO2 inhibitor', 
                                    '100uM TDO2 inhibitor (680C91) TDO2 inhibitor 680C91 (Tocris Bioscience, Bristol, UK) was dissolved in dimethyl sulfoxide (DMSO). TDO2 inhibitor / DMSO were added to the yeast extract in the diet to a final concentration of 100 µM (680C91) . Flies were introduced to the inhibitor-supplemented media upon eclosion; supplemented media were changed every other day. Files were snaped freeze in liquid nitrogen at the age of 21 days for further process.':'100uM TDO2 inhibitor',
                                    'All the samples had been under DMBA/TPA treatment for more than 24 weeks to induce skin tumors.':np.nan, 
                                    'We firstly generated isogenic human embryonic stem cell (hESC) lines with SIRT6-deficiency from wild-type(WT) hESCs by knocking out the 1st exon of SIRT6 gene via a TALEN-mediated approach. Then hMSCs were differentiated from hESCs, and maintained in human MSC culture medium by serial passaging.':np.nan, 
                                    'No No treatment':'Control', 
                                    'saline':'Saline',
                                    'Tumors from mice were surgically removed, flash-frozen in liquid nitrogen and stored at -80 °C.':np.nan,
                                    'siCtrl Chondrocytes were nucleofected with siRNA to Sirt6 or an empty vector control (AMAXA nucleofection method) for 72 hours':np.nan, 
                                    'siSirt6 Chondrocytes were nucleofected with siRNA to Sirt6 or an empty vector control (AMAXA nucleofection method) for 72 hours':np.nan,
                                    'MEF cells were generated from wildtype, Sirt6-/-, P53-/-, or Atm-/-P53-/- mice.':np.nan, 
                                    'Wild-type':np.nan, 
                                    'siSirt6': np.nan,
                                    'SIRT6-Knock Out':np.nan, 
                                    'NSCs proliferation medium':np.nan, 
                                    'All the samples had been under DMBA/TPA treatment for more than 24 weeks to induce skin tumors.':'DMBA/TPA', 
                                    'FLAG-tagged empty vector, WT, K3R, or K3Q SIRT6 were stably overexpressed in A549 and H1299 cells via lentivirus transduction.':np.nan,
                                    'Lentiviral ShRNA transduction':np.nan,
                                    'treated with DMSO (1/1000) 4 days':'DMSO (1/1000) 4 days', 
                                    'treated with 2µM of RAFi 4 days':'2µM of RAFi 4 days', 
                                    'treated with 100nM of RAFi and 1nM of MEKi 4 days':'100nM of RAFi and 1nM of MEKi 4 days', 
                                    'HUVECs at 100% confluence were infected with control adenoviral LacZ or adenoviral SIRT6 for 48 h.':np.nan
                                   })

In [110]:
pheno_data_clean['treatment'] = pheno_data_clean.treatment.replace({'dSirt6 and dMyc were overexpressed using the GAL4-UAS system ':'',
                                    'Mice were treated with with an isocaloric control diet or a 5% Lieber-DeCarli diet for 10 days plus a single binge of maltose dextrin or ethanol \(5 g/kg\) ':''
}, regex=True)

<>:2: SyntaxWarning: invalid escape sequence '\('
<>:2: SyntaxWarning: invalid escape sequence '\('
/var/folders/kb/2390td2n4f3bqhz9pvyr26w00000gn/T/ipykernel_29931/4104991924.py:2: SyntaxWarning: invalid escape sequence '\('
  'Mice were treated with with an isocaloric control diet or a 5% Lieber-DeCarli diet for 10 days plus a single binge of maltose dextrin or ethanol \(5 g/kg\) ':''


In [111]:
pheno_data_clean['treatment'] = pheno_data_clean['treatment'].replace({'Chondrocytes were nucleofected with siRNA to Sirt6 or an empty vector control (AMAXA nucleofection method) for 72 hours siCtrl': np.nan, 
                                                                           'Chondrocytes were nucleofected with siRNA to Sirt6 or an empty vector control (AMAXA nucleofection method) for 72 hours siSirt6': np.nan})

In [112]:
pheno_data_clean['treatment'].unique()

<ArrowStringArray>
[                             'Control',
                                    nan,
                                  'LPS',
                               'Saline',
                               'Ang II',
                             'pair-fed',
                          'ethanol-fed',
                             'DMBA/TPA',
                 'DMSO (1/1000) 4 days',
                   '2µM of RAFi 4 days',
 '100nM of RAFi and 1nM of MEKi 4 days',
                      'Adult (10 days)',
                       'Aged (40 days)',
         'dMyc + dSirt6 overexpression',
                  'DMSO TDO2 inhibitor',
                 '100uM TDO2 inhibitor']
Length: 16, dtype: str

In [113]:
pheno_data_clean['tissue'] = pheno_data_clean['tissue'].str.capitalize()

In [114]:
pheno_data_clean = (
    pheno_data_clean
    .assign(treatment = lambda df: np.where(
        pheno_data_clean.treatment.isna(),
        df.treatment.replace({np.nan: 'Control'}),
        df.treatment
    ))
)

In [115]:
pheno_data_clean = pheno_data_clean.loc[~pheno_data_clean.relation.isin(['SRX13459876', 'SRX13459877'])]

## Summary

In [116]:
from skrub import TableReport

In [117]:
TableReport(pheno_data_clean)

Processing column  22 / 22


,,,,,,,,,,,,,,,,,,,,,,


## Save data

In [118]:
pheno_data_clean.to_csv("./SIRT6_datasets_metadata.csv")

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
if pheno_data_parrow:
    pq.write_to_dataset(
        pheno_data_parrow,
        root_path='sirt6_metadata/',
        partition_cols=['Organism', "Series_id"]
    )

In [ ]:
df = pd.read_parquet(
    'sirt6_metadata/', 
    filters=[('Organism', '==', 'Sus scrofa')]
)

In [ ]:
df.columns

Index(['Geo_accession', 'Submission_date', 'Extract_protocol',
       'Contact_institute', 'Instrument_model', 'Library_strategy', 'Relation',
       'Title', 'Summary', 'Growth_protocol', 'Time', 'Passages', 'Batch',
       'Sex', 'Genotype', 'Treatment', 'Tissue', 'Cell type', 'Strain', 'Age',
       'Condition', 'Submission_year', 'Organism', 'Series_id'],
      dtype='object')

In [ ]:
df.head()

,Geo_accession,Submission_date,Extract_protocol,Contact_institute,Instrument_model,Library_strategy,Relation,Title,Summary,Growth_protocol,...,Genotype,Treatment,Tissue,Cell type,Strain,Age,Condition,Submission_year,Organism,Series_id
0,GSM4889180,Nov 09 2020,The cell was lysised and the RNA was amplifica...,"College of Animal Science and Technology, Nanj...",HiSeq X Ten,RNA-Seq,SRX9461172,SIRT6 maintains redox homeostasis to promote p...,"SIRT6, the sixth member of sirtuin family prot...",None,...,WT,Control,oocyte,,,Metaphase II,Control,2020,Sus scrofa,GSE161068
1,GSM4889181,Nov 09 2020,The cell was lysised and the RNA was amplifica...,"College of Animal Science and Technology, Nanj...",HiSeq X Ten,RNA-Seq,SRX9461173,SIRT6 maintains redox homeostasis to promote p...,"SIRT6, the sixth member of sirtuin family prot...",None,...,WT,Control,oocyte,,,Metaphase II,Control,2020,Sus scrofa,GSE161068
2,GSM4889182,Nov 09 2020,The cell was lysised and the RNA was amplifica...,"College of Animal Science and Technology, Nanj...",HiSeq X Ten,RNA-Seq,SRX9461174,SIRT6 maintains redox homeostasis to promote p...,"SIRT6, the sixth member of sirtuin family prot...",None,...,SIRT6-KO,Control,oocyte,,,Metaphase II,Control,2020,Sus scrofa,GSE161068
3,GSM4889183,Nov 09 2020,The cell was lysised and the RNA was amplifica...,"College of Animal Science and Technology, Nanj...",HiSeq X Ten,RNA-Seq,SRX9461171,SIRT6 maintains redox homeostasis to promote p...,"SIRT6, the sixth member of sirtuin family prot...",None,...,SIRT6-KO,Control,oocyte,,,Metaphase II,Control,2020,Sus scrofa,GSE161068


## Analytics

In [ ]:
import plotly.express as px

In [ ]:
import plotly.io as pio
pio.renderers.default = 'iframe'

In [ ]:
species_pheno_data['Submission_year'] = species_pheno_data.Submission_date.str.split(' ').str[2].astype('int')

In [ ]:
grouped_data = species_pheno_data.groupby(['Submission_year', 'Organism', 'Series_id']).size().reset_index(name = 'Sample size')

In [ ]:
grouped_data

,Submission_year,Organism,Series_id,Sample size
0,2015,Homo sapiens,GSE64642,8
1,2017,Homo sapiens,GSE102813,18
2,2017,Macaca fascicularis,GSE102830,64
3,2018,Mus musculus,GSE109280,6
4,2018,Mus musculus,GSE115953,15
5,2019,Mus musculus,GSE129370,12
6,2019,Mus musculus,"GSE130690,GSE130692",6
7,2020,Mus musculus,GSE157838,28
8,2020,Sus scrofa,GSE161068,4
9,2021,Drosophila melanogaster,GSE191320,16


In [ ]:
fig = px.scatter(pd.DataFrame(grouped_data), x = "Submission_year", y = "Sample size", 
                 size = "Sample size", color = "Organism",
                 hover_name = "Series_id", size_max = 60)
fig.show()